last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


In [3]:
import os


In [4]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [5]:
#! curl ipinfo.io

In [6]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
)

In [7]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 35.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 48.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [9]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

In [10]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [11]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [12]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [13]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")

data = loader.load()

In [14]:
data[:3]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [15]:
from langchain_community.document_loaders import WebBaseLoader

In [16]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

#print(documents[0].page_content)

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [17]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [18]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

In [19]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [20]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [21]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [22]:
!pip install tiktoken

In [23]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [24]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [25]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [26]:
client = openai.OpenAI()

In [29]:
import os
import openai

# 환경 변수에서 키를 가져올 때 .strip()을 붙여서 \n 이나 \r을 제거합니다.
api_key = os.getenv("OPENAI_API_KEY").strip()

client = openai.OpenAI(api_key=api_key)

# 그 후 기존 코드를 다시 실행해 보세요.
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [30]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [32]:
!curl ipinfo.io

{
  "ip": "34.75.183.55",
  "hostname": "55.183.75.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}

In [36]:
#400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

! pip install -q sentence_transformers

from langchain.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

ImportError: cannot import name 'HuggingFaceEmbeddings' from 'langchain.embeddings' (/usr/local/lib/python3.12/dist-packages/langchain/embeddings/__init__.py)

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [38]:
import os
import openai

# 환경 변수에서 가져올 때 맨 뒤의 줄바꿈(\n)을 확실하게 제거합니다.
api_key = os.getenv("OPENAI_API_KEY").strip()

# 또는 만약 코랩 Secrets(열쇠 아이콘)를 쓰신다면 아래처럼 해주세요.
# from google.colab import userdata
# api_key = userdata.get('OPENAI_API_KEY').strip()

In [39]:
# 이 코드가 있는 블록을 찾아서 다시 실행해 주세요!
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(api_key=api_key)

In [40]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [41]:
print(embeddings[1])

[-0.014838632196187973, -0.02360120788216591, 0.017914321273565292, 0.0023915055207908154, 0.0006661378429271281, -0.006050948053598404, -0.020751487463712692, -0.005209841299802065, -0.011417712084949017, -0.024279115721583366, -0.012823741883039474, 0.002093351911753416, -0.01787666045129299, 0.008141161873936653, -0.006967377848923206, 0.016633830964565277, 0.03545202687382698, -0.030832218006253242, 0.015353339724242687, -0.010036790743470192, -0.012993218377232552, 0.00864331517368555, 0.001070998958311975, -0.005012118257582188, -0.0035495967604219913, 0.01788921467959881, 0.006176486611366272, -0.019558873027563095, 0.014386693947017193, 0.010218821465969086, 0.017399614676833153, -0.017826445400714874, 0.0064526707865297794, -0.012892788276076317, -0.004638641607016325, -0.015667185187339783, 0.011938696727156639, 0.015780169516801834, 0.0026347360108047724, -0.013909648172557354, -0.004867749288678169, -0.0006618224433623254, 0.005194148980081081, -0.011317281983792782, -0.014

In [42]:
len(embeddings[1])

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [43]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [44]:
query = ["this is red fruit"]

In [45]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.9336998751036216
0.8637313695098962
0.844670115963403


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [1]:
!pip install chromadb

In [2]:
!pip install langchain-chroma

In [3]:
from langchain_chroma import Chroma

In [4]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [8]:
# 1. tiktoken_len 함수 정의를 맨 위에 추가합니다.
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)


# 2. 기존 코드 (이전 단계에서 추가한 import 포함)
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

# 이제 tiktoken_len이 정상적으로 인식됩니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [9]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [31]:
import os
import re
import shutil
from google.colab import userdata
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

print("Chroma에 임베딩 시작")

# 1. OpenAI API Key 불러오기
my_api_key = userdata.get("OPENAI_API_KEY")

if my_api_key is None:
    raise ValueError("Colab 보안 비밀에 OPENAI_API_KEY가 없습니다.")

# 2. 공백, 줄바꿈, 따옴표 제거
my_api_key = re.sub(r"\s+", "", my_api_key)
my_api_key = my_api_key.strip().strip('"').strip("'")

if my_api_key.startswith("OPENAI_API_KEY="):
    my_api_key = my_api_key.replace("OPENAI_API_KEY=", "", 1)

if not my_api_key.startswith("sk-"):
    raise ValueError("API 키 형식이 이상합니다. sk- 로 시작해야 합니다.")

os.environ["OPENAI_API_KEY"] = my_api_key

# 3. OpenAI 임베딩 모델 생성
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=my_api_key
)

# 4. 임베딩 차원 확인
test_vector = embedding_model.embed_query("dimension test")
print("현재 임베딩 차원:", len(test_vector))

# 5. 기존 Chroma DB 삭제
CHROMA_DIR = "/content/chroma_openai_1536"

if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)
    print("기존 Chroma DB 삭제 완료")

# 6. 새 Chroma DB 생성
db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="day1_rag_openai_1536",
    persist_directory=CHROMA_DIR
)

print("Chroma DB 생성 완료")
print("저장된 문서 수:", db._collection.count())


Chroma에 임베딩 시작
현재 임베딩 차원: 1536
Chroma DB 생성 완료
저장된 문서 수: 4


이제 쿼리를 날려보겠습니다

In [32]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [33]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [34]:
!pip install -U langchain langchain-classic

In [35]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [38]:
!pip install -qU langchain-openai langchain-core

In [39]:
import os
import re
from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI API Key 불러오기
my_api_key = userdata.get("OPENAI_API_KEY")

if my_api_key is None:
    raise ValueError("Colab 보안 비밀에 OPENAI_API_KEY가 없습니다.")

# 공백, 줄바꿈, 따옴표 제거
my_api_key = re.sub(r"\s+", "", my_api_key)
my_api_key = my_api_key.strip().strip('"').strip("'")

if my_api_key.startswith("OPENAI_API_KEY="):
    my_api_key = my_api_key.replace("OPENAI_API_KEY=", "", 1)

if not my_api_key.startswith("sk-"):
    raise ValueError("API 키 형식이 이상합니다. sk- 로 시작해야 합니다.")

os.environ["OPENAI_API_KEY"] = my_api_key

# OpenAI Chat 모델 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

print("ChatOpenAI 모델 준비 완료")

ChatOpenAI 모델 준비 완료


체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [40]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [41]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_11064/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


Demian is described as having a face that is not distinctly masculine or feminine, but rather timeless and different from others. The narrator perceives him as having an elegant and at-ease demeanor, with a deep, quiet, almost fanatical yet passionate absorption in his gaze. His face seems to carry the marks of various periods of history, making him appear unearthly or spirit-like. Overall, he is portrayed as being unimaginably different from those around him, evoking a sense of attraction and repulsion simultaneously.

마크다운 형식으로 출력해봅니다

In [42]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is not distinctly masculine or feminine, but rather timeless and different from others. The narrator perceives him as having an elegant and at-ease demeanor, with a deep, quiet, almost fanatical yet passionate absorption in his gaze. His face seems to carry the marks of various periods of history, making him appear unearthly or spirit-like. Overall, he is portrayed as being unimaginably different from those around him, evoking a sense of attraction and repulsion simultaneously.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [43]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


"Demian" is a novel by Hermann Hesse, first published in 1919. It is a coming-of-age story about a boy named Emil Sinclair and his journey toward self-discovery, heavily influenced by his mysterious friend, Max Demian. Since "Demian" is a literary work, the characters are described through text without specific visual depictions. Readers often imagine Emil Sinclair and Max Demian based on their interpretations of the text and the themes conveyed in the novel.

Each reader might visualize them differently based on personal imagination and Hesse's descriptions, focusing on aspects like Demian's enigmatic presence or Sinclair's evolving perspective. If you're referring to visual adaptations or artistic portrayals, those can vary widely depending on the illustrator or filmmaker's interpretation.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

In [44]:
# 필요한 라이브러리 설치
!pip install -qU langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters langchain-classic chromadb pypdf tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.8 MB/s eta 0:00:00


Text splitter 사용을 위한 준비입니다

In [45]:
# 공통 import 및 보조 함수 준비
import os
import re
import shutil
from pathlib import Path

import tiktoken
from google.colab import userdata
from IPython.display import Markdown, display

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

try:
    # 최신 LangChain 환경
    from langchain_classic.chains.retrieval_qa.base import RetrievalQA
except Exception:
    # 일부 구버전 환경
    from langchain.chains import RetrievalQA


def load_openai_api_key() -> str:
    """
    Colab 보안 비밀에서 OpenAI API Key를 안전하게 불러옵니다.
    - 권장 이름: OPENAI_API_KEY
    - 기존 실습에서 OPENAI_KEY로 저장한 경우도 보조적으로 지원합니다.
    - 줄바꿈/공백/따옴표/OPENAI_API_KEY= 접두어 문제를 제거합니다.
    """
    key = userdata.get("OPENAI_API_KEY") or userdata.get("OPENAI_KEY")

    if key is None:
        raise ValueError(
            "Colab 왼쪽 열쇠 아이콘(보안 비밀)에 OPENAI_API_KEY를 추가해주세요."
        )

    key = re.sub(r"\s+", "", key)
    key = key.strip().strip('"').strip("'")

    if key.startswith("OPENAI_API_KEY="):
        key = key.replace("OPENAI_API_KEY=", "", 1)

    if not key.startswith("sk-"):
        raise ValueError("API 키 형식이 이상합니다. sk- 로 시작해야 합니다.")

    os.environ["OPENAI_API_KEY"] = key
    return key


# tiktoken 기반 토큰 길이 계산 함수
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text: str) -> int:
    return len(tokenizer.encode(text))


OPENAI_API_KEY = load_openai_api_key()

print("준비 완료")
print("API key loaded:", OPENAI_API_KEY[:10] + "..." + OPENAI_API_KEY[-4:])

준비 완료
API key loaded: sk-proj-WI...N1IA


### Step 1 Document loader

In [46]:
# Step 1. Document Loader
# Demian PDF를 Document 객체로 불러옵니다.

PDF_PATH = "/content/Demian.pdf"

if not Path(PDF_PATH).exists():
    raise FileNotFoundError(
        f"{PDF_PATH} 파일이 없습니다. Colab 왼쪽 파일 탭에 Demian.pdf를 업로드한 뒤 다시 실행하세요."
    )

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print("로드된 페이지 수:", len(pages))
print("첫 페이지 metadata:", pages[0].metadata)
print("첫 페이지 미리보기:")
print(pages[0].page_content[:500])

로드된 페이지 수: 182
첫 페이지 metadata: {'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}
첫 페이지 미리보기:
DEMIAN 
• 
Downloaded from https://www.holybooks.com


### Step 2 Text splitters

In [47]:
# Step 2. Text Splitters
# 긴 PDF 문서를 RAG 검색에 적합한 chunk 단위로 나눕니다.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          # 최대 500 tokens 정도로 분할
    chunk_overlap=50,        # 앞뒤 문맥 보존을 위해 50 tokens 겹침
    length_function=tiktoken_len,
    separators=["\n\n", "\n", ".", " ", ""],
)

docs = text_splitter.split_documents(pages)

print("생성된 chunk 수:", len(docs))
print("첫 번째 chunk token 길이:", tiktoken_len(docs[0].page_content))
print("첫 번째 chunk 미리보기:")
print(docs[0].page_content[:500])

생성된 chunk 수: 182
첫 번째 chunk token 길이: 16
첫 번째 chunk 미리보기:
DEMIAN 
• 
Downloaded from https://www.holybooks.com


### Step 3 Vector Empeddings

In [48]:
# Step 3. Vector Embeddings
# 각 chunk를 OpenAI 임베딩 벡터로 변환할 모델을 준비합니다.
# text-embedding-3-small은 1536차원 벡터를 생성합니다.

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY,
)

test_embedding = embedding_model.embed_query("Demian RAG embedding dimension test")

print("임베딩 모델 준비 완료")
print("임베딩 차원:", len(test_embedding))

임베딩 모델 준비 완료
임베딩 차원: 1536


In [49]:
# Step 3-2. Chroma VectorStore 생성
# 이전 실습에서 만든 768차원 Chroma DB가 남아 있으면 차원 불일치 오류가 발생할 수 있습니다.
# 따라서 이번 완성 예제는 별도 폴더를 사용하고, 실행할 때마다 깨끗하게 다시 만듭니다.

CHROMA_DIR = "/content/chroma_demian_openai_1536"
COLLECTION_NAME = "demian_rag_openai_1536"

if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)
    print("기존 Chroma DB 삭제 완료:", CHROMA_DIR)

db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

print("Chroma VectorStore 생성 완료")
print("저장된 chunk 수:", db._collection.count())

Chroma VectorStore 생성 완료
저장된 chunk 수: 182


### Step 4 Retrievers

In [50]:
# Step 4. Retriever
# MMR 검색은 비슷한 chunk만 반복해서 가져오는 문제를 줄이고, 다양한 근거를 확보하는 데 유리합니다.

retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,        # 최종적으로 LLM에게 전달할 chunk 수
        "fetch_k": 10  # 먼저 넓게 가져온 뒤 MMR로 3개를 선별
    },
)

print("Retriever 생성 완료")

Retriever 생성 완료


In [51]:
# Retriever가 실제로 관련 문서를 잘 가져오는지 먼저 확인합니다.

test_query = "How does Demian look like?"
retrieved_docs = retriever.invoke(test_query)

print("검색된 문서 수:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    page = doc.metadata.get("page", "unknown")
    source = doc.metadata.get("source", "unknown")
    print(f"\n--- Retrieved chunk {i} | page: {page} | source: {source} ---")
    print(doc.page_content[:700])

검색된 문서 수: 3

--- Retrieved chunk 1 | page: 53 | source: /content/Demian.pdf ---
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time 

--- Retrieved chunk 2 | page: 70 | source: /content/Demian.pdf ---
THE THIEF 
eyes were fixed on that pale, stone mask, speJibound, and 
I felt that here was the real Demian I What he had been 
before when he had gone 

### Step 5 Question Answering

In [52]:
# Step 5. Question Answering
# Retriever가 찾은 근거 문서를 LLM에 넣어 답변을 생성하는 RAG QA Chain을 만듭니다.

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=OPENAI_API_KEY,
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

print("RAG QA Chain 생성 완료")

RAG QA Chain 생성 완료


In [53]:
# 완성 예제 실행
# RAG 답변과 근거 문서 page를 함께 출력합니다.

def ask_rag(question: str):
    result = qa_chain.invoke({"query": question})

    answer = result["result"]
    source_documents = result["source_documents"]

    display(Markdown(f"## 질문\n{question}"))
    display(Markdown(f"## RAG 답변\n{answer}"))

    print("\n[근거 문서]")
    for i, doc in enumerate(source_documents, start=1):
        page = doc.metadata.get("page", "unknown")
        source = doc.metadata.get("source", "unknown")
        preview = doc.page_content.replace("\n", " ")[:300]
        print(f"{i}. page={page}, source={source}")
        print(f"   {preview}...")

    return result


# 예시 질문 1: 인물 묘사
result_1 = ask_rag("How does Demian look like?")

# 예시 질문 2: 주제 이해
result_2 = ask_rag("What is the main theme of Demian?")

## 질문
How does Demian look like?

## RAG 답변
Demian's appearance is described as elegant and at ease, with a face that seems neither masculine nor childish, but rather timeless and bearing elements of both. The narrator perceives something different about him, almost a feminine quality, and notes that his face could be seen as handsome or attractive, yet also potentially repelling. Overall, he is depicted as unimaginarily different from others, with an aura that suggests he is like an animal, a spirit, or an image.


[근거 문서]
1. page=53, source=/content/Demian.pdf
   DEMIAN  with a feeling of nausea, I noticed Demian's expression.  He had not thrust himself to the front but stood right  at the back, looking elc!gant and at ease as usual. His  glance seemed directed at the horse's head, and again it  . showed that deep, quiet, almost fanatical yet passionate  abs...
2. page=70, source=/content/Demian.pdf
   THE THIEF  eyes were fixed on that pale, stone mask, speJibound, and  I felt that here was the real Demian I What he had been  before when he had gone around and chatted with me  was only IJ,alf of him, a person who for a time was play­ ing a part, adapting himself to it, joining in the game  to obl...
3. page=33, source=/content/Demian.pdf
   DEMIAN  character and have some significance. But I merely knew  that Demian's mother was reported to be very wealthy.  It was also said that neither she nor her son ever  attended church. One boy wondered whether they might  not be Jews but they could equa

## 질문
What is the main theme of Demian?

## RAG 답변
The main theme of "Demian" revolves around the exploration of self-discovery, individuality, and the struggle between societal norms and personal identity. The novel delves into the idea that true understanding of oneself and one's purpose often lies outside the constraints of conventional society, represented by communities, states, and religious institutions. It emphasizes the importance of embracing one's inner self and the journey towards personal freedom, often through the influence of significant individuals who challenge societal expectations. The narrative also touches on the duality of human nature, the conflict between good and evil, and the quest for authenticity in a world that often suppresses it.


[근거 문서]
1. page=149, source=/content/Demian.pdf
   DEMIAN  will be revealed. It will then become clear that the will  of humanity is never and nowhere to be identified with  that of our present communities, states and nations,  clubs and churches. No; what nature wants of man is  written in a few individuals,· in you, in me. It was  written in Chris...
2. page=47, source=/content/Demian.pdf
   DEMIAN  me. AB it was, I was clinging with all my roots to my  former earthly paradise; I had returned home and had  been accepted in grace. Jklt it was not Demian's world  nor was he- suited to it. He too-though in a different  way from Kromer-was a 'tempter' and moreover my  link with the second, ...
3. page=70, source=/content/Demian.pdf
   THE THIEF  eyes were fixed on that pale, stone mask, speJibound, and  I felt that here was the real Demian I What he had been  before when he had gone around and chatted with me  was only IJ,alf of him, a person who for a time was play­ ing a part, adaptin